In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sample_data_generator import load_sample_data

# Retrieve Data

In [2]:
data = np.load('./data/energy_market_samples.npz')

In [3]:
data['data'][0]

array([[ 44.24,   8.26,  56.69],
       [ 41.69,  30.11,  76.7 ],
       [ 45.4 , -21.67, 117.17],
       [ 43.03,  33.77,  98.37],
       [ 45.47,  33.88, 102.74],
       [ 53.65,  17.85,  51.46],
       [ 46.9 ,  41.11,  63.6 ],
       [ 58.03,  36.05,  77.73],
       [ 60.87,  26.28,   0.  ],
       [ 64.39,  54.  ,  22.9 ],
       [ 43.06,   8.92,  43.3 ],
       [ 43.04,  14.83,  50.62],
       [ 42.82,  18.39,  18.34],
       [ 40.56,  36.77,  92.38],
       [ 40.75,  23.07,  99.92],
       [-21.86,   5.23, 146.06],
       [ 32.11, -23.8 , 129.71],
       [ 53.94,  22.05,  57.76],
       [ 52.54,  40.31,  54.2 ],
       [ 48.15,  30.5 ,  34.91],
       [ 52.92,  46.16,   1.  ],
       [ 37.92,  23.78,  58.64],
       [ 39.26,  32.93,  58.98],
       [ 41.01,  24.63,  49.01]])

In [4]:
data['target_names']

array(['dalmp', 'rtlmp', 'wind_power_mw'], dtype='<U13')

# Visualization

In [5]:
data["data"].mean(axis=0)

array([[ 37.7331,  55.5887,  62.2152],
       [ 36.7059,  18.0166,  84.4636],
       [ 37.5745,  14.2057, 102.089 ],
       [ 35.7912,  15.174 , 105.363 ],
       [ 38.2773,  18.7907,  93.0133],
       [ 41.3033,  23.8484,  76.9461],
       [ 42.3351,  27.4911,  53.2092],
       [ 52.2107,  88.0791,  32.449 ],
       [ 52.1561, 125.3576,  21.1455],
       [ 50.8013,  95.4033,  19.8273],
       [ 41.2522,  32.8969,  26.1025],
       [ 39.0064,  98.0919,  46.0963],
       [ 37.1152,  20.2992,  72.1504],
       [ 33.5588,  14.1185,  92.9045],
       [ 33.4135,  15.9508, 101.1187],
       [ 30.7663,   9.7744, 105.3749],
       [ 32.5438,  13.1343,  84.5255],
       [ 41.6009,  26.9966,  67.1174],
       [ 44.9768,  31.276 ,  42.9374],
       [ 45.5259, 100.5625,  26.1909],
       [ 47.1764, 157.0679,  17.1542],
       [ 37.2148, 106.1772,  22.2113],
       [ 37.5965,  29.4331,  35.1768],
       [ 36.6864,  21.2216,  63.9099]])

## Scenarios

In [6]:
def plot_scenarios(column:str):
    fig = go.Figure()
    mean_data = data['data'].mean(axis=0)
    median_data = np.median(data['data'], axis=0)
    data_names = {str(name): i for i, name in enumerate(data['target_names'])}
    print(data_names)
    unit = "MW" if column == "wind_power_mw" else "$/MWh"

    for i, scenario in enumerate(data['data']):
        y = scenario[:, data_names[column]]
        x = np.arange(len(y))
        fig.add_trace(
            go.Scatter(
                x=x,
                y=y ,
                line_shape='hv',
                name= f"scenario {i+1} {column.upper()} "
            )
        )
    fig.update_traces({'line': {'color': "lightgrey"}})

    fig.add_trace(
        go.Scatter(
            x = x,
            y = mean_data[:, data_names[column]],
            line_shape='hv',
            name= f"{column.upper()} mean"
        )
    )
    fig.update_layout(
        title = f"Scenarios and mean {column.upper()}",
        xaxis_title = "Time[hours]",
        yaxis_title = f"{column.title()}[{unit}]"
    )

    fig.show()

In [7]:
plot_scenarios("wind_power_mw")
plot_scenarios("dalmp")
plot_scenarios("rtlmp")

{'dalmp': 0, 'rtlmp': 1, 'wind_power_mw': 2}


{'dalmp': 0, 'rtlmp': 1, 'wind_power_mw': 2}


{'dalmp': 0, 'rtlmp': 1, 'wind_power_mw': 2}


## Spread

In [8]:
spreads = data['data'][:,:,0] - data['data'][:,:,1]

fig = go.Figure()
for i, spread in enumerate(spreads):
    x = np.arange(len(spread))
    fig.add_trace(
        go.Scatter(
            x=x,
            y=spread,
            line_shape='hv',
            name= f"scenario {i+1} spread"
        )
    )
fig.update_traces({'line': {"color": "lightgray"}})
fig.update_layout(
    title=f"Difference between Day Ahead and Real Time Locational Marginal Price per Scenario",
    xaxis_title="Time[Hours]",
    yaxis_title="Spread(DALMP - RTLMP)[$/MWh]"
)
fig.show()

## Correlation

In [9]:
import pandas as pd
import matplotlib.pyplot as plt

flat_data = data['data'].reshape(-1, 3)  # Flatten across scenarios and hours
df = pd.DataFrame(flat_data, columns=data['target_names'])
correlation_matrix = df.corr()

fig = go.Figure()
fig.add_trace(
    go.Heatmap(
        x = correlation_matrix.columns,
        y = correlation_matrix.index,
        z = correlation_matrix.values,
        colorscale='RdBu'
    )
)
fig.show()

There seems to be an inverse relationship between wind and the real time price such that when wind generation is low then the real time price will be high.

## Questions
For ensemble scenarios are all scenarios equally likely?  
Doesn't this concept of buy back suggest that we are constrained to buy back based on how much we sold into the market in the first place?  
Is there a way to limit how much i buy based on this calculation of the cleared price?  


## Deliverables and Time Management
Python Implementation: Given the 2-3 hour time frame, aim for a minimum viable solution with a clear problem formulation and optimization strategy.

Technical Report: Briefly describe your model, results, and any insights from the synthetic data. Mention potential real-world extensions.

Development Plan: Clearly define objectives, constraints, and potential improvements. Discuss any trade-offs or limitations you identified.



: 